In [2]:
# Step 1: Import Necessary Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Step 2: Load the Dataset
# Ensure you have 'diabetes.csv' uploaded to your Colab environment
# You can download it from Kaggle (UCI Pima Indians Diabetes Database)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)

print("Original Dataset Shape:", df.shape)

# Step 3: Data Preprocessing (The Zero-Value Strategy from the Paper)
# The paper states that zeros in certain columns are meaningless.
columns_with_invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# We replace zeros with NaN temporarily to make calculations easier
df[columns_with_invalid_zeros] = df[columns_with_invalid_zeros].replace(0, np.nan)

# The paper's specific strategy: replace missing values with the median/mean
# calculated separately for diabetic (Outcome=1) and healthy (Outcome=0) patients.
def impute_missing_by_class(dataframe, columns):
    for col in columns:
        # Calculate medians for both classes (median is often preferred for skewed health data like Insulin)
        median_healthy = dataframe[dataframe['Outcome'] == 0][col].median()
        median_diabetic = dataframe[dataframe['Outcome'] == 1][col].median()

        # Apply the replacement
        dataframe.loc[(dataframe['Outcome'] == 0) & (dataframe[col].isnull()), col] = median_healthy
        dataframe.loc[(dataframe['Outcome'] == 1) & (dataframe[col].isnull()), col] = median_diabetic

    return dataframe

df_cleaned = impute_missing_by_class(df.copy(), columns_with_invalid_zeros)

# Step 4: Train-Test Split
# The paper used a 70% training/validation and 30% testing split
X = df_cleaned.drop('Outcome', axis=1)
y = df_cleaned['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

# --- Model 2: K-Nearest Neighbors (KNN) ---
# The paper tests KNN, which classifies data based on the closest training examples
knn_model = KNeighborsClassifier(n_neighbors=5) # 5 is a standard starting point for 'K'
knn_model.fit(X_train, y_train)
knn_predictions = knn_model.predict(X_test)

# --- Model 3: Support Vector Machine (SVM) ---
# The paper notes SVM tries to locate an ideal hyperplane to separate the classes [cite: 191]
svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)

# --- Model 4: Decision Tree (DT) ---
# Uses a top-down approach to categorize the dataset [cite: 233]
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)

# Step 6: Evaluation and Classification Reports

print("\n" + "="*40)
print("K-NEAREST NEIGHBORS (KNN) REPORT")
print("="*40)
print(classification_report(y_test, knn_predictions))
print(f"Accuracy: {accuracy_score(y_test, knn_predictions):.4f}")

print("\n" + "="*40)
print("RANDOM FOREST CLASSIFICATION REPORT")
print("="*40)
print(classification_report(y_test, rf_predictions))
print(f"Accuracy: {accuracy_score(y_test, rf_predictions):.4f}")

print("\n" + "="*40)
print("SUPPORT VECTOR MACHINE (SVM) REPORT")
print("="*40)
print(classification_report(y_test, svm_predictions))
print(f"Accuracy: {accuracy_score(y_test, svm_predictions):.4f}")

print("\n" + "="*40)
print("DECISION TREE (DT) REPORT")
print("="*40)
print(classification_report(y_test, dt_predictions))
print(f"Accuracy: {accuracy_score(y_test, dt_predictions):.4f}")

Original Dataset Shape: (768, 9)

Training Data Shape: (537, 8)
Testing Data Shape: (231, 8)

K-NEAREST NEIGHBORS (KNN) REPORT
              precision    recall  f1-score   support

           0       0.87      0.92      0.90       150
           1       0.84      0.75      0.79        81

    accuracy                           0.86       231
   macro avg       0.85      0.84      0.84       231
weighted avg       0.86      0.86      0.86       231

Accuracy: 0.8615

RANDOM FOREST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.88      0.92      0.90       150
           1       0.84      0.78      0.81        81

    accuracy                           0.87       231
   macro avg       0.86      0.85      0.85       231
weighted avg       0.87      0.87      0.87       231

Accuracy: 0.8701

SUPPORT VECTOR MACHINE (SVM) REPORT
              precision    recall  f1-score   support

           0       0.90      0.83      0.87       150
  